In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q
!pip install numpy==1.26.4 -q
!pip install basicsr==1.4.2 -q

import torchvision.transforms.functional as F
import sys
sys.modules['torchvision.transforms.functional_tensor'] = F

!pip install realesrgan -q
!mkdir -p /content/weights
!wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth -O /content/weights/RealESRGAN_x2plus.pth
print('done')

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

input_dir = '/content/sharp-frames'
if not os.path.exists(input_dir):
    for root, dirs, fls in os.walk('/content/'):
        if any(f.startswith('frame_') for f in fls):
            input_dir = root
            break

frames = sorted([f for f in os.listdir(input_dir) if f.endswith('.jpg')])
print(f'{len(frames)} frames found')

In [ ]:
import cv2
import torch
import os
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer
from tqdm import tqdm

model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=2)
upsampler = RealESRGANer(
    scale=2,
    model_path='/content/weights/RealESRGAN_x2plus.pth',
    model=model,
    tile=400,
    tile_pad=10,
    pre_pad=0,
    half=True
)

output_dir = '/content/ai-frames'
os.makedirs(output_dir, exist_ok=True)

print(f'GPU: {torch.cuda.get_device_name(0)}')

for fname in tqdm(frames):
    out_path = os.path.join(output_dir, fname)
    if os.path.exists(out_path):
        continue
    img = cv2.imread(os.path.join(input_dir, fname), cv2.IMREAD_UNCHANGED)
    output, _ = upsampler.enhance(img, outscale=2)
    cv2.imwrite(out_path, output, [cv2.IMWRITE_JPEG_QUALITY, 95])

print(f'{len(os.listdir(output_dir))} frames upscaled')

In [ ]:
import shutil, os
from google.colab import files

shutil.make_archive('/content/ai-frames', 'zip', '/content', 'ai-frames')
print(f"{os.path.getsize('/content/ai-frames.zip') / 1024 / 1024:.1f} MB")
files.download('/content/ai-frames.zip')